# BaseballCV — Stolen Base Analysis

End-to-end pipeline for stolen-base clips:

1. **Harvest** — find stolen-base plays in MLB's statsapi live feed and download their broadcast clips from Baseball Savant (uses the `from_play_ids` constructor added in [PR #181](https://github.com/BaseballCV/BaseballCV/pull/181)).
2. **Detect bases** — run the new `base_detector` model ([PR #179](https://github.com/BaseballCV/BaseballCV/pull/179)) frame-by-frame to locate the target base.
3. **Track the runner** — measure the runner-to-base distance over time to see the lead, the jump, and the slide.

> **Note:** requires `baseballcv` 0.1.27+ (the scraper 403 fix).

**Runtime:** `Runtime → Change runtime type → GPU` recommended; CPU works for a few clips.

In [ ]:
# Install BaseballCV (SB scraping needs >= 0.1.27)
!pip install -q "baseballcv>=0.1.27"

In [ ]:
import os
import requests
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

from baseballcv.functions import LoadTools, BaseballSavVideoScraper

# --- Pipeline parameters ---
START_DATE = "2026-08-01"   # statcast era (2015+) only
END_DATE   = "2026-08-03"
TEAM_ABBR  = None           # e.g. "NYY" to restrict to one team
MAX_PLAYS  = 12             # SB/CS plays to harvest before stopping
MAX_CLIPS  = 3              # clips to actually download (keep it polite)

CLIPS_DIR = "sb_clips"
os.makedirs(CLIPS_DIR, exist_ok=True)

load_tools = LoadTools()
print("Setup OK")

## 1. Harvest stolen-base plays

Successful steals are **not** result-level plays in the statsapi feed — they are *action events* inside `playEvents`, tagged `details.eventType = stolen_base_2b/3b/home` (plus `caught_stealing_*`), each carrying an `actionPlayId` that maps 1:1 to a Savant video.

In [ ]:
SCHEDULE_URL = "https://statsapi.mlb.com/api/v1/schedule"     # v1 for schedule
LIVE_FEED_URL = "https://statsapi.mlb.com/api/v1.1/game/{}/feed/live"  # v1.1 for feeds
SB_EVENTS = ("stolen_base_2b", "stolen_base_3b", "stolen_base_home",
             "caught_stealing_2b", "caught_stealing_3b", "caught_stealing_home")

def harvest_sb_plays(start_date, end_date, team_abbr=None, max_plays=MAX_PLAYS) -> pd.DataFrame:
    sched = requests.get(SCHEDULE_URL, timeout=30, params={
        "sportId": 1, "startDate": start_date, "endDate": end_date, "gameType": "R"}).json()
    games = [g for d in sched.get("dates", []) for g in d.get("games", [])
             if team_abbr is None or team_abbr in {
                 g["teams"]["home"]["team"].get("abbreviation"),
                 g["teams"]["away"]["team"].get("abbreviation")}]

    rows = []
    for g in games:
        if len(rows) >= max_plays:
            break
        live = requests.get(LIVE_FEED_URL.format(g["gamePk"]), timeout=30).json()
        for play in live["liveData"]["plays"]["allPlays"]:
            for e in play.get("playEvents", []):
                et = e.get("details", {}).get("eventType", "")
                if et in SB_EVENTS and e.get("actionPlayId"):
                    rows.append({
                        "game_pk": g["gamePk"],
                        "play_id": e["actionPlayId"],
                        "event": et,
                        "description": e["details"]["description"],
                        "inning": play["about"]["inning"],
                        "half": play["about"]["halfInning"],
                    })
            if len(rows) >= max_plays:
                break
    return pd.DataFrame(rows[:max_plays])

sb_plays = harvest_sb_plays(START_DATE, END_DATE, TEAM_ABBR)
if sb_plays.empty:
    raise RuntimeError("No stolen-base plays found — widen the date range or clear TEAM_ABBR.")
sb_plays

In [ ]:
to_download = sb_plays.head(MAX_CLIPS)
scraper = BaseballSavVideoScraper.from_play_ids(list(to_download["play_id"]),
                                                game_pks=list(to_download["game_pk"]),
                                                download_folder=CLIPS_DIR)
scraper.run_executor()

clips = sorted(os.path.join(CLIPS_DIR, f) for f in os.listdir(CLIPS_DIR) if f.endswith(".mp4"))
assert clips, "No clips downloaded — check the scraper logs above."
clips

## 2. Detect the target base with `base_detector`

Savant SB clips are usually the straight-away center-field broadcast view, so the base being stolen stays near mid-frame. We sample frames, keep the highest-confidence detection per frame, and follow it over time.

In [ ]:
base_model = YOLO(load_tools.load_model("base_detector"))
print("Classes:", base_model.names)

def sample_frames(video_path, max_frames=40):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    idxs = np.linspace(0, max(total - 1, 0), min(max_frames, total)).astype(int)
    frames = []
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, frame = cap.read()
        if ok:
            frames.append((i / fps, frame))
    cap.release()
    return frames

def detect(model, frame, conf=0.25):
    res = model.predict(frame, conf=conf, verbose=False)[0]
    out = []
    for b in res.boxes:
        x1, y1, x2, y2 = b.xyxy[0].tolist()
        out.append({"conf": float(b.conf[0]),
                    "cx": (x1 + x2) / 2, "cy": (y1 + y2) / 2,
                    "xyxy": (x1, y1, x2, y2)})
    return out

CLIP = clips[0]
frames = sample_frames(CLIP)
detections = [detect(base_model, f) for _, f in frames]
print(f"{os.path.basename(CLIP)}: {len(frames)} frames sampled, "
      f"{sum(len(d) > 0 for d in detections)} with a base detection")

In [ ]:
# Annotated frames + the best-confidence base track over time
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, (t, frame), det in zip(axes.flat, frames[:: len(frames) // 8], detections[:: len(detections) // 8]):
    vis = frame.copy()
    for d in det:
        x1, y1, x2, y2 = map(int, d["xyxy"])
        cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 220, 0), 2)
        cv2.putText(vis, f"base {d['conf']:.2f}", (x1, y1 - 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 220, 0), 2)
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(f"t={t:.1f}s", fontsize=9)
    ax.axis("off")
plt.suptitle(os.path.basename(CLIP), y=0.98)
plt.tight_layout()
plt.show()

track = [(t, max(d, key=lambda x: x["conf"])) for (t, _), d in zip(frames, detections) if d]
if track:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    ax1.plot([t for t, _ in track], [d["cx"] for _, d in track])
    ax1.set_xlabel("time (s)"); ax1.set_ylabel("base x (px)"); ax1.set_title("target base — horizontal position")
    ax2.plot([t for t, _ in track], [d["conf"] for _, d in track])
    ax2.set_xlabel("time (s)"); ax2.set_ylabel("confidence"); ax2.set_title("target base — detection confidence")
    plt.tight_layout()
    plt.show()
else:
    print("No base detections in this clip — try another one (CLIP = clips[1]) or lower conf.")

## 3. Runner-to-base distance over time

A stock `yolov8n-pose` gives the runner's position (highest-confidence person in frame). The pixel distance from the runner's feet to the target base traces the whole story: the lead, the jump, and the slide. Broadcast clips often **cut cameras** mid-play, which teleports the runner in pixel space — the plot below detects those cuts and only measures the closest approach within the final continuous shot. Values are in **pixels**; converting to feet needs a calibration step (e.g. homography from the base paths), listed as a next step.

In [ ]:
pose_model = YOLO("yolov8n-pose.pt")

def runner_feet(frame, conf=0.3):
    res = pose_model.predict(frame, conf=conf, verbose=False)[0]
    if not len(res.boxes):
        return None
    i = int(res.boxes.conf.argmax())          # biggest-confidence person = the runner
    x1, y1, x2, y2 = res.boxes.xyxy[i].tolist()
    return (x1 + x2) / 2, y2                  # feet center

if track:
    base_px = max((d for _, d in track), key=lambda d: d["conf"])  # most confident view of the base
    curve = []
    for t, f in frames:
        pos = runner_feet(f)
        if pos is not None:
            curve.append((t, pos, float(np.hypot(pos[0] - base_px["cx"], pos[1] - base_px["cy"]))))

    if curve:
        # Mark camera cuts so the dips after them are read as replays, not as the arrival
        width = frames[0][1].shape[1]
        cuts = [i for i in range(1, len(curve))
                if abs(curve[i][1][0] - curve[i - 1][1][0]) > 0.25 * width]

        ts = [t for t, _, _ in curve]
        dist = [d for _, _, d in curve]
        best = int(np.argmin(dist))           # closest approach = the runner reaching the base

        plt.figure(figsize=(12, 4))
        plt.plot(ts, dist)
        for c in cuts:
            plt.axvline(curve[c][0], color="red", ls="--", lw=1)
        plt.plot(ts[best], dist[best], "o", color="orange")
        plt.xlabel("time (s)"); plt.ylabel("runner→base distance (px)")
        plt.title(f"{os.path.basename(CLIP)} — closest approach: {dist[best]:.0f}px at t={ts[best]:.1f}s "
                  f"(red dashes = camera cuts; dips after a cut are replays)")
        plt.tight_layout()
        plt.show()
    else:
        print("No runner detected — the camera may be too tight on the base.")

## Wrap-up

| Step | Tool | Status |
|---|---|---|
| Find SB plays | statsapi live feed (`actionPlayId`) | ✅ |
| Download clips | `BaseballSavVideoScraper.from_play_ids` (#181) | ✅ |
| Detect bases | `base_detector` (#179) | ✅ |
| Runner tracking | `yolov8n-pose` (stock ultralytics) | ✅ |

Next steps:
- **Calibration**: homography from the base paths (90 ft squares) to convert pixels → feet, giving true lead distance and jump timing.
- **Batch mode**: run the distance curve over every harvested clip and rank the biggest jumps.
- **Swap in a trained runner model**: a dedicated player detector would beat the stock pose model on wide outfield shots.